# BalanceLens — Notebook-First AI Decision Support (Free + Local)

**What this notebook gives you**

- A working **Retrieval-Augmented Generation (RAG)** pipeline
- An open-source **LLM reasoning layer** (runs locally via Hugging Face `transformers`)
- A curated, editable knowledge base on:
  - burnout prevention and recovery
  - boundary setting (work / peers / professors)
  - sleep and stress management
  - academic resilience and workload planning
- Structured outputs:
  - situation interpretation
  - competing pressures
  - realistic short-term actions
  - boundary-setting scripts
  - weekly balance plan
- A simple in-notebook **Gradio UI**

**Important safety note (read)**

BalanceLens is **not** a medical/mental health professional service and does not replace therapy, counseling, or emergency support.

If you are in immediate danger or considering self-harm, contact local emergency services right now. If you're in the U.S., you can call/text **988** (Suicide & Crisis Lifeline). If you're outside the U.S., look up your country's crisis line.


In [ ]:
# If you're running in Jupyter, you can install dependencies here.
# Recommended: create/activate a virtual environment for this project.

import sys

print("Python:", sys.version)


In [ ]:
# Install free, open-source dependencies
# Notes:
# - We try FAISS first, but on Windows + Python 3.12 it may not have wheels.
# - If FAISS isn't available, we'll fall back to hnswlib (also free/local) automatically.

%pip -q install "sentence-transformers>=3.0.0" "transformers>=4.41.0" "accelerate>=0.31.0" "torch" "numpy" "pydantic>=2.0" "gradio>=4.0" "hnswlib>=0.8.0"

In [ ]:
# Optional: attempt to install FAISS (if wheels exist for your platform)
# If this fails, don't worry — the notebook will automatically use hnswlib.

try:
    import faiss  # noqa: F401
    print("FAISS already available.")
except Exception:
    try:
        %pip -q install faiss-cpu
        import faiss  # noqa: F401
        print("Installed faiss-cpu successfully.")
    except Exception as e:
        print("FAISS not available on this environment. We'll use hnswlib instead.")
        print("Details:", repr(e))


In [ ]:
# LangChain RAG stack (free)
%pip -q install "langchain>=0.2.0" "langchain-community>=0.2.0" "langchain-text-splitters>=0.2.0" "fastapi>=0.110" "uvicorn[standard]>=0.29" "python-multipart>=0.0.9" "pypdf>=4.0.0"

In [ ]:
from __future__ import annotations

import json
import os
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np

PROJECT_ROOT = Path(r"d:\ChiEAC\BalanceLens")
DATA_DIR = PROJECT_ROOT / "data"
KNOWLEDGE_DIR = DATA_DIR / "knowledge"
INDEX_DIR = DATA_DIR / "index"

for p in [DATA_DIR, KNOWLEDGE_DIR, INDEX_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_ROOT)
print("Knowledge dir:", KNOWLEDGE_DIR)
print("Index dir:", INDEX_DIR)


In [ ]:
# Create a curated knowledge base (editable markdown files)
# You can replace/extend these with your own curated sources later.

SEED_DOCS: Dict[str, str] = {
    "burnout_and_recovery.md": """# Burnout risk & recovery

## What burnout can look like (non-clinical)
- Feeling emotionally drained, cynical, or numb
- Noticing your work/academic output is dropping despite trying harder
- Sleep feels unrefreshing; chronic fatigue
- Increased irritability or tears; low patience

## Early warning signs
- Skipping meals, hydration, or movement because you 'can't afford the time'
- Needing caffeine to function most days
- Falling behind and then using all-nighters to catch up
- Losing joy in things that normally help you recover

## What helps quickly (this week)
- Protect one basic need first: sleep window, meals, or a short walk
- Reduce load by one notch: fewer shifts, fewer commitments, or smaller deliverables
- Use a 'minimum viable week': define what must be done vs. what can wait

## If you keep hitting the same wall
- Consider talking to a campus counselor, advisor, or trusted adult
- Ask for accommodations/support when you have a documented pattern of overload
""",
    "boundary_setting_scripts.md": """# Boundary setting (work, professors, friends)

## Principles
- Be clear, brief, and kind.
- Name the constraint (time/health/academic deadline) without over-explaining.
- Offer one alternative (different time, smaller scope, or next week).

## Work: fewer shifts
"""Example"""
Hi [Manager Name] — I can’t take extra shifts this week because of school deadlines. I can do my scheduled shifts on [days]. If you need coverage next week, I can revisit on [date].
"""

## Professor: asking for support
"""Example"""
Hello Professor [Name], I’m experiencing a high workload this week and I’m concerned about meeting the deadline for [assignment]. Could we discuss an extension or alternative plan? I can submit a partial draft by [date] and complete the rest by [date].
"""

## Friends/peers: saying no
"""Example"""
I care about you, but I’m at capacity this week. I can’t help with that right now. If it’s still needed, I can check in on [day/time].
"""
""",
    "sleep_stress_management.md": """# Sleep and stress basics (practical)

## If sleep is short
- Aim for a consistent wake time; use a 30–60 minute wind-down
- Avoid heavy work in bed (protect 'bed = sleep')
- If you must study late, do a 10-minute shutdown routine: list tasks, set next action, close laptop

## Micro-recovery (5–15 minutes)
- Walk outside, hydration + snack, stretching, 4-7-8 breathing
- One small 'completion': tidy desk, send one email, outline one page

## When stress spikes
- Name what’s urgent vs important
- Choose one next action that reduces pressure (email, reschedule, ask for help)
- Avoid stacking self-criticism on top of stress
""",
    "decision_tradeoffs_framework.md": """# Decision tradeoffs framework

## Identify the pressures
- Academic: exams, labs, assignments, attendance policies
- Work: shifts, manager expectations, income needs
- Health: sleep, nutrition, medical needs, mental bandwidth
- Relationships: partner, friends, family expectations
- Finances: bills, debt, savings, emergencies

## A quick 4-box check
1) What happens if I say yes?
2) What happens if I say no?
3) What’s the smallest 'yes' that still helps?
4) What boundary protects my future self?

## Good short-term choices
- Reduce harm even if you can’t optimize everything
- Make a decision that you can sustain for 7 days
- Prefer choices that increase future options (sleep, asking early, protecting exam prep)
""",
}

for name, content in SEED_DOCS.items():
    path = KNOWLEDGE_DIR / name
    if not path.exists():
        path.write_text(content, encoding="utf-8")

print("Knowledge files:")
for p in sorted(KNOWLEDGE_DIR.glob("*.md")):
    print(" -", p.name)


In [ ]:
# Ingestion + chunking + embedding + vector indexing (FAISS via LangChain)

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings


def load_markdown_docs(dir_path: Path):
    docs = []
    for md in sorted(dir_path.glob("*.md")):
        loader = TextLoader(str(md), encoding="utf-8")
        loaded = loader.load()
        for d in loaded:
            d.metadata.update({"source": md.name, "path": str(md)})
        docs.extend(loaded)
    return docs


raw_docs = load_markdown_docs(KNOWLEDGE_DIR)
print("Loaded docs:", len(raw_docs))
print("Example metadata:", raw_docs[0].metadata if raw_docs else None)

splitter = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=150)
chunks = splitter.split_documents(raw_docs)
print("Chunks:", len(chunks))

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)
print("Embeddings model:", EMBED_MODEL)


In [ ]:
# Build and persist the vector index

from langchain_community.vectorstores import FAISS

INDEX_NAME = "balancelens_faiss"
INDEX_PATH = INDEX_DIR / INDEX_NAME

vectorstore = FAISS.from_documents(chunks, embedding=embeddings)
vectorstore.save_local(str(INDEX_PATH))

print("Saved index to:", INDEX_PATH)


In [ ]:
# Retrieval test

loaded_vs = FAISS.load_local(str(INDEX_PATH), embeddings, allow_dangerous_deserialization=True)
retriever = loaded_vs.as_retriever(search_kwargs={"k": 4})

query = "My manager is asking me to take extra shifts but I have exams and I'm exhausted"
results = retriever.get_relevant_documents(query)

for i, d in enumerate(results, 1):
    print(f"\n--- Result {i} | source={d.metadata.get('source')} ---\n")
    print(d.page_content[:500])


In [ ]:
# Open-source LLM reasoning layer (free/local)
# We use a small instruct-tuned model that runs on CPU.

import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline

LLM_MODEL = "google/flan-t5-base"  # lightweight and free

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(LLM_MODEL)

gen = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
)

print("Loaded LLM:", LLM_MODEL)
print("Torch:", torch.__version__)


In [ ]:
# Structured response schema

from pydantic import BaseModel, Field


class BoundaryScript(BaseModel):
    audience: str = Field(..., description="Who the script is for, e.g. manager/professor/friend")
    script: str = Field(..., description="A short message the user can copy")


class BalanceLensResponse(BaseModel):
    situation_summary: str
    likely_pressures: List[str]
    what_might_help_this_week: List[str]
    boundary_scripts: List[BoundaryScript]
    weekly_balance_plan: Dict[str, str]
    citations: List[Dict[str, str]]
    safety_note: str


In [ ]:
# RAG + prompt assembly and JSON parsing

import re


def _compact_sources(docs):
    out = []
    for d in docs:
        out.append(
            {
                "source": d.metadata.get("source", "unknown"),
                "excerpt": re.sub(r"\s+", " ", d.page_content.strip())[:550],
            }
        )
    return out


def _extract_json(text: str) -> str:
    # Try to find the first JSON object in the model output
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        raise ValueError("No JSON object found")
    return text[start : end + 1]


SYSTEM_GUARDRAILS = (
    "You are BalanceLens, a practical decision-support assistant for college-aged women. "
    "You do NOT provide medical diagnosis. You do NOT shame the user. "
    "You prioritize safety, sleep, and sustainable choices. "
    "You must produce valid JSON that matches the schema."
)


def generate_balancelens_response(user_text: str, k: int = 4) -> BalanceLensResponse:
    docs = retriever.get_relevant_documents(user_text)[:k]
    sources = _compact_sources(docs)

    context = "\n\n".join(
        [f"[Source: {s['source']}]\n{s['excerpt']}" for s in sources]
    )

    schema_hint = {
        "situation_summary": "string",
        "likely_pressures": [
            "academic overload",
            "burnout risk",
            "financial pressure",
            "people-pleasing",
            "sleep debt",
        ],
        "what_might_help_this_week": ["string", "string"],
        "boundary_scripts": [
            {"audience": "manager", "script": "string"},
            {"audience": "professor", "script": "string"},
        ],
        "weekly_balance_plan": {
            "current_pressure_points": "string",
            "what_to_protect_first": "string",
            "one_boundary_to_set": "string",
            "one_recovery_action": "string",
            "one_achievable_academic_step": "string",
        },
        "citations": [{"source": "string", "excerpt": "string"}],
        "safety_note": "string",
    }

    prompt = f"""{SYSTEM_GUARDRAILS}

User situation:
{user_text}

Retrieved knowledge (use this to ground your advice):
{context}

Return ONLY JSON. Match this shape:
{json.dumps(schema_hint, ensure_ascii=False)}

Rules:
- Keep advice realistic for this week.
- Include at least 2 boundary scripts tailored to the situation.
- Add citations using the retrieved sources.
- Include a brief safety note encouraging professional/campus support if stress is overwhelming.
"""

    raw = gen(prompt)[0]["generated_text"]

    try:
        obj = json.loads(_extract_json(raw))
    except Exception:
        # repair attempt
        repair = gen(
            "Fix this into valid JSON only (no markdown).\n\n" + raw
        )[0]["generated_text"]
        obj = json.loads(_extract_json(repair))

    # Force citations to the actual retrieved sources
    obj["citations"] = sources

    return BalanceLensResponse.model_validate(obj)


In [ ]:
# End-to-end test

example = "I have three exams this week but my manager keeps asking me to take extra shifts. I'm exhausted and feel guilty saying no."
resp = generate_balancelens_response(example)

print(resp.model_dump_json(indent=2))


In [ ]:
# In-notebook UI (Gradio)

import gradio as gr


def ui_run(user_text: str):
    r = generate_balancelens_response(user_text)
    d = r.model_dump()
    # Make it readable
    return (
        d["situation_summary"],
        "\n".join(f"- {x}" for x in d["likely_pressures"]),
        "\n".join(f"- {x}" for x in d["what_might_help_this_week"]),
        "\n\n".join(
            f"**{s['audience']}**\n{s['script']}" for s in d["boundary_scripts"]
        ),
        json.dumps(d["weekly_balance_plan"], indent=2),
        "\n\n".join(
            f"- {c['source']}: {c['excerpt']}" for c in d["citations"]
        ),
        d["safety_note"],
    )


demo = gr.Interface(
    fn=ui_run,
    inputs=gr.Textbox(lines=6, label="Describe your situation"),
    outputs=[
        gr.Textbox(label="Situation interpretation"),
        gr.Textbox(label="Likely pressures"),
        gr.Textbox(label="What might help this week"),
        gr.Markdown(label="Boundary scripts"),
        gr.Code(label="Weekly balance plan (JSON)", language="json"),
        gr.Textbox(label="Citations (retrieved excerpts)", lines=6),
        gr.Textbox(label="Safety note"),
    ],
    title="BalanceLens (Local, Free RAG)",
    description="Decision-support with retrieval-grounded guidance (not a replacement for professional support).",
)

demo.launch(share=False)


## Generate a real FastAPI backend + Next.js frontend + Docker deploy files

The next cells will **write project files** into your `BalanceLens/` folder:

- `backend/` (FastAPI) — loads the saved FAISS index and serves `/chat`, `/health`
- `frontend/` (Next.js) — responsive mobile-first UI that calls the backend
- `docker-compose.yml` — runs both

Everything is still authored here in the notebook (so you have the full code in `.ipynb`), but we generate actual runnable files for deployment.


In [ ]:
from textwrap import dedent

BACKEND_DIR = PROJECT_ROOT / "backend"
FRONTEND_DIR = PROJECT_ROOT / "frontend"

BACKEND_DIR.mkdir(parents=True, exist_ok=True)

print("Will write backend to:", BACKEND_DIR)
print("Will write frontend to:", FRONTEND_DIR)


In [ ]:
# Write backend files (FastAPI)

backend_main = dedent(f"""
from __future__ import annotations

import json
import os
import re
from pathlib import Path
from typing import Dict, List

from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline


PROJECT_ROOT = Path(__file__).resolve().parents[1]
INDEX_PATH = PROJECT_ROOT / "data" / "index" / "balancelens_faiss"

EMBED_MODEL = os.getenv("EMBED_MODEL", "sentence-transformers/all-MiniLM-L6-v2")
LLM_MODEL = os.getenv("LLM_MODEL", "google/flan-t5-base")
K = int(os.getenv("RAG_K", "4"))


class ChatRequest(BaseModel):
    text: str = Field(..., min_length=2)


class BoundaryScript(BaseModel):
    audience: str
    script: str


class ChatResponse(BaseModel):
    situation_summary: str
    likely_pressures: List[str]
    what_might_help_this_week: List[str]
    boundary_scripts: List[BoundaryScript]
    weekly_balance_plan: Dict[str, str]
    citations: List[Dict[str, str]]
    safety_note: str


def _extract_json(text: str) -> str:
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        raise ValueError("No JSON found")
    return text[start : end + 1]


def _compact_sources(docs):
    out = []
    for d in docs:
        out.append(
            {{
                "source": d.metadata.get("source", "unknown"),
                "excerpt": re.sub(r"\s+", " ", d.page_content.strip())[:550],
            }}
        )
    return out


SYSTEM_GUARDRAILS = (
    "You are BalanceLens, a practical decision-support assistant for college-aged women. "
    "You do NOT provide medical diagnosis. You do NOT shame the user. "
    "You prioritize safety, sleep, and sustainable choices. "
    "You must produce valid JSON that matches the schema."
)


app = FastAPI(title="BalanceLens API", version="0.1.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


@app.on_event("startup")
def _startup():
    global embeddings, vs, retriever, gen

    embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)
    vs = FAISS.load_local(str(INDEX_PATH), embeddings, allow_dangerous_deserialization=True)
    retriever = vs.as_retriever(search_kwargs={{"k": K}})

    tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
    model = AutoModelForSeq2SeqLM.from_pretrained(LLM_MODEL)
    gen = pipeline(
        "text2text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=512,
    )


@app.get("/health")
def health():
    return {{"ok": True}}


@app.post("/chat", response_model=ChatResponse)
def chat(req: ChatRequest):
    docs = retriever.get_relevant_documents(req.text)
    sources = _compact_sources(docs)
    context = "\n\n".join([f"[Source: {{s['source']}}]\n{{s['excerpt']}}" for s in sources])

    schema_hint = {{
        "situation_summary": "string",
        "likely_pressures": ["string"],
        "what_might_help_this_week": ["string"],
        "boundary_scripts": [{{"audience": "manager", "script": "string"}}],
        "weekly_balance_plan": {{
            "current_pressure_points": "string",
            "what_to_protect_first": "string",
            "one_boundary_to_set": "string",
            "one_recovery_action": "string",
            "one_achievable_academic_step": "string",
        }},
        "citations": [{{"source": "string", "excerpt": "string"}}],
        "safety_note": "string",
    }}

    prompt = f"""{{SYSTEM_GUARDRAILS}}

User situation:
{{req.text}}

Retrieved knowledge (use this to ground your advice):
{{context}}

Return ONLY JSON. Match this shape:
{{json.dumps(schema_hint, ensure_ascii=False)}}

Rules:
- Keep advice realistic for this week.
- Include at least 2 boundary scripts tailored to the situation.
- Add citations using the retrieved sources.
- Include a brief safety note encouraging professional/campus support if stress is overwhelming.
"""

    raw = gen(prompt)[0]["generated_text"]

    try:
        obj = json.loads(_extract_json(raw))
    except Exception:
        repair = gen("Fix into valid JSON only (no markdown).\n\n" + raw)[0]["generated_text"]
        obj = json.loads(_extract_json(repair))

    obj["citations"] = sources
    return obj
""")

(BACKEND_DIR / "main.py").write_text(backend_main, encoding="utf-8")

(BACKEND_DIR / "requirements.txt").write_text(
    dedent(
        """
        fastapi
        uvicorn[standard]
        pydantic
        langchain
        langchain-community
        langchain-text-splitters
        sentence-transformers
        transformers
        accelerate
        torch
        faiss-cpu
        """
    ).strip()
    + "\n",
    encoding="utf-8",
)

print("Wrote:", BACKEND_DIR / "main.py")
print("Wrote:", BACKEND_DIR / "requirements.txt")


In [ ]:
# Create Next.js frontend (mobile-first) and write the app code
# This uses `create-next-app` (free). If you re-run, it won't overwrite unless you delete the folder.

import subprocess

if not FRONTEND_DIR.exists():
    subprocess.check_call(
        [
            "npx",
            "create-next-app@latest",
            str(FRONTEND_DIR),
            "--ts",
            "--eslint",
            "--app",
            "--src-dir",
            "--no-tailwind",
            "--use-npm",
        ]
    )
    print("Created Next.js app:", FRONTEND_DIR)
else:
    print("Next.js app already exists:", FRONTEND_DIR)


In [ ]:
# Write frontend files (App Router) — responsive, mobile-first UI

src_app = FRONTEND_DIR / "src" / "app"
src_app.mkdir(parents=True, exist_ok=True)

(src_app / "globals.css").write_text(
    dedent(
        """
        :root {
          --bg: #0b1220;
          --card: #121b2f;
          --text: #e9eefc;
          --muted: #b8c2e6;
          --accent: #7c5cff;
          --danger: #ff5c7a;
          --border: rgba(255, 255, 255, 0.10);
        }
        * { box-sizing: border-box; }
        html, body { height: 100%; }
        body {
          margin: 0;
          font-family: ui-sans-serif, system-ui, -apple-system, Segoe UI, Roboto, Arial, "Apple Color Emoji", "Segoe UI Emoji";
          background: radial-gradient(1200px 800px at 20% 0%, rgba(124,92,255,0.25), transparent 60%),
                      radial-gradient(900px 700px at 80% 10%, rgba(255,92,122,0.18), transparent 55%),
                      var(--bg);
          color: var(--text);
        }
        a { color: inherit; }
        .container {
          width: 100%;
          max-width: 980px;
          margin: 0 auto;
          padding: 20px 16px 56px;
        }
        .header {
          display: flex;
          flex-direction: column;
          gap: 10px;
          margin: 8px 0 18px;
        }
        .title {
          font-size: 28px;
          font-weight: 800;
          letter-spacing: -0.02em;
          margin: 0;
        }
        .subtitle {
          margin: 0;
          color: var(--muted);
          line-height: 1.4;
          font-size: 14px;
        }
        .grid {
          display: grid;
          grid-template-columns: 1fr;
          gap: 14px;
        }
        @media (min-width: 860px) {
          .grid { grid-template-columns: 1fr 1fr; }
        }
        .card {
          background: rgba(18, 27, 47, 0.78);
          border: 1px solid var(--border);
          border-radius: 16px;
          padding: 16px;
          backdrop-filter: blur(10px);
        }
        .card h2 {
          font-size: 14px;
          letter-spacing: 0.02em;
          text-transform: uppercase;
          color: var(--muted);
          margin: 0 0 10px;
        }
        textarea {
          width: 100%;
          min-height: 160px;
          resize: vertical;
          border-radius: 14px;
          border: 1px solid var(--border);
          padding: 12px 12px;
          background: rgba(11, 18, 32, 0.55);
          color: var(--text);
          outline: none;
          line-height: 1.4;
        }
        textarea:focus { border-color: rgba(124,92,255,0.6); }
        .row {
          display: flex;
          flex-wrap: wrap;
          gap: 10px;
          align-items: center;
          margin-top: 10px;
        }
        button {
          appearance: none;
          border: 0;
          border-radius: 999px;
          padding: 10px 14px;
          background: linear-gradient(135deg, rgba(124,92,255,0.95), rgba(255,92,122,0.85));
          color: white;
          font-weight: 700;
          cursor: pointer;
        }
        button:disabled {
          opacity: 0.6;
          cursor: not-allowed;
        }
        .pill {
          border: 1px solid var(--border);
          border-radius: 999px;
          padding: 8px 10px;
          color: var(--muted);
          font-size: 12px;
        }
        .mono { font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, "Liberation Mono", "Courier New", monospace; }
        .kvs { display: grid; gap: 8px; }
        .kv { display: grid; gap: 6px; }
        .kv .k { color: var(--muted); font-size: 12px; text-transform: uppercase; letter-spacing: 0.02em; }
        .kv .v { white-space: pre-wrap; line-height: 1.45; }
        .list { margin: 0; padding-left: 18px; }
        .danger {
          border-left: 4px solid var(--danger);
          padding-left: 10px;
          color: var(--muted);
          line-height: 1.4;
          font-size: 13px;
        }
        """
    ).lstrip(),
    encoding="utf-8",
)

(src_app / "layout.tsx").write_text(
    dedent(
        """
        import "./globals.css";
        import type { Metadata } from "next";

        export const metadata: Metadata = {
          title: "BalanceLens",
          description: "AI-powered decision support with RAG (free + local).",
        };

        export default function RootLayout({
          children,
        }: {
          children: React.ReactNode;
        }) {
          return (
            <html lang="en">
              <body>{children}</body>
            </html>
          );
        }
        """
    ).lstrip(),
    encoding="utf-8",
)

(src_app / "page.tsx").write_text(
    dedent(
        """
        "use client";

        import { useMemo, useState } from "react";

        type BoundaryScript = { audience: string; script: string };
        type ChatResponse = {
          situation_summary: string;
          likely_pressures: string[];
          what_might_help_this_week: string[];
          boundary_scripts: BoundaryScript[];
          weekly_balance_plan: Record<string, string>;
          citations: { source: string; excerpt: string }[];
          safety_note: string;
        };

        const DEFAULT_TEXT =
          "I have three exams this week but my manager keeps asking me to take extra shifts. I'm exhausted and feel guilty saying no.";

        export default function Page() {
          const [text, setText] = useState(DEFAULT_TEXT);
          const [loading, setLoading] = useState(false);
          const [error, setError] = useState<string | null>(null);
          const [data, setData] = useState<ChatResponse | null>(null);

          const apiBase = useMemo(() => {
            return process.env.NEXT_PUBLIC_API_BASE ?? "http://localhost:8000";
          }, []);

          async function onRun() {
            setLoading(true);
            setError(null);
            setData(null);
            try {
              const res = await fetch(`${apiBase}/chat`, {
                method: "POST",
                headers: { "Content-Type": "application/json" },
                body: JSON.stringify({ text }),
              });
              if (!res.ok) {
                const msg = await res.text();
                throw new Error(msg || `HTTP ${res.status}`);
              }
              const json = (await res.json()) as ChatResponse;
              setData(json);
            } catch (e: any) {
              setError(e?.message ?? String(e));
            } finally {
              setLoading(false);
            }
          }

          return (
            <main className="container">
              <header className="header">
                <h1 className="title">BalanceLens</h1>
                <p className="subtitle">
                  A retrieval-grounded decision companion for everyday overload (not a
                  replacement for professional support).
                </p>
                <div className="danger">
                  If you feel unsafe or in immediate danger, contact local emergency
                  services. In the U.S., call/text <b>988</b>.
                </div>
              </header>

              <section className="card">
                <h2>Your situation</h2>
                <textarea
                  value={text}
                  onChange={(e) => setText(e.target.value)}
                  placeholder="Describe what's going on…"
                />
                <div className="row">
                  <button onClick={onRun} disabled={loading || text.trim().length < 2}>
                    {loading ? "Thinking…" : "Get grounded guidance"}
                  </button>
                  <span className="pill mono">API: {apiBase}</span>
                </div>
                {error ? (
                  <p className="subtitle" style={{ color: "#ffb4c2" }}>
                    Error: {error}
                  </p>
                ) : null}
              </section>

              {data ? (
                <section className="grid" style={{ marginTop: 14 }}>
                  <div className="card">
                    <h2>Situation interpretation</h2>
                    <div className="kvs">
                      <div className="kv">
                        <div className="v">{data.situation_summary}</div>
                      </div>
                    </div>
                  </div>

                  <div className="card">
                    <h2>Competing pressures</h2>
                    <ul className="list">
                      {data.likely_pressures.map((p, i) => (
                        <li key={i}>{p}</li>
                      ))}
                    </ul>
                  </div>

                  <div className="card">
                    <h2>What might help this week</h2>
                    <ul className="list">
                      {data.what_might_help_this_week.map((x, i) => (
                        <li key={i}>{x}</li>
                      ))}
                    </ul>
                  </div>

                  <div className="card">
                    <h2>Boundary scripts</h2>
                    <div className="kvs">
                      {data.boundary_scripts.map((s, i) => (
                        <div key={i} className="kv">
                          <div className="k">{s.audience}</div>
                          <div className="v">{s.script}</div>
                        </div>
                      ))}
                    </div>
                  </div>

                  <div className="card">
                    <h2>Weekly balance plan</h2>
                    <div className="kvs">
                      {Object.entries(data.weekly_balance_plan).map(([k, v]) => (
                        <div key={k} className="kv">
                          <div className="k">{k.replaceAll("_", " ")}</div>
                          <div className="v">{v}</div>
                        </div>
                      ))}
                    </div>
                  </div>

                  <div className="card">
                    <h2>Grounding (retrieved excerpts)</h2>
                    <div className="kvs">
                      {data.citations.map((c, i) => (
                        <div key={i} className="kv">
                          <div className="k">{c.source}</div>
                          <div className="v">{c.excerpt}</div>
                        </div>
                      ))}
                    </div>
                  </div>

                  <div className="card">
                    <h2>Safety note</h2>
                    <div className="v">{data.safety_note}</div>
                  </div>
                </section>
              ) : null}
            </main>
          );
        }
        """
    ).lstrip(),
    encoding="utf-8",
)

# Point frontend to backend base URL
(FRONTEND_DIR / ".env.local").write_text(
    "NEXT_PUBLIC_API_BASE=http://localhost:8000\n",
    encoding="utf-8",
)

print("Wrote frontend app files.")


In [ ]:
# Docker deployment (free) — Linux container ensures FAISS availability

(PROJECT_ROOT / "docker-compose.yml").write_text(
    dedent(
        """
        services:
          backend:
            build:
              context: ./backend
            ports:
              - "8000:8000"
            environment:
              - EMBED_MODEL=sentence-transformers/all-MiniLM-L6-v2
              - LLM_MODEL=google/flan-t5-base
              - RAG_K=4
            volumes:
              - ./data:/app/data

          frontend:
            build:
              context: ./frontend
            ports:
              - "3000:3000"
            environment:
              - NEXT_PUBLIC_API_BASE=http://localhost:8000
            depends_on:
              - backend
        """
    ).lstrip(),
    encoding="utf-8",
)

(BACKEND_DIR / "Dockerfile").write_text(
    dedent(
        """
        FROM python:3.11-slim

        WORKDIR /app

        # System deps
        RUN apt-get update && apt-get install -y --no-install-recommends \
            build-essential \
            && rm -rf /var/lib/apt/lists/*

        COPY requirements.txt /app/requirements.txt
        RUN pip install --no-cache-dir -r /app/requirements.txt

        COPY . /app

        EXPOSE 8000
        CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
        """
    ).lstrip(),
    encoding="utf-8",
)

(FRONTEND_DIR / "Dockerfile").write_text(
    dedent(
        """
        FROM node:20-slim

        WORKDIR /app
        COPY package*.json /app/
        RUN npm ci
        COPY . /app
        RUN npm run build

        EXPOSE 3000
        CMD ["npm", "run", "start", "--", "-p", "3000"]
        """
    ).lstrip(),
    encoding="utf-8",
)

print("Wrote docker-compose.yml and Dockerfiles.")


## How to run locally

### Option A — Notebook UI (fastest)
Run cells top-to-bottom, then use the Gradio app.

### Option B — Full web app (Next.js + FastAPI)
From a terminal in `d:\ChiEAC\BalanceLens`:

```bash
# 1) Build the index (run the ingestion cells above at least once)

# 2) Backend
python -m venv .venv
# activate on Windows PowerShell:
.\.venv\Scripts\Activate.ps1
pip install -r backend\requirements.txt
uvicorn backend.main:app --reload --port 8000

# 3) Frontend (new terminal)
cd frontend
npm install
npm run dev
```

Open `http://localhost:3000`.

## How to run with Docker (recommended for deployment)

```bash
docker compose up --build
```

Open `http://localhost:3000`.

## Deployment notes (Railway / AWS)

- **Railway**: deploy `backend/` as a Docker service (it exposes port 8000). Set a persistent volume for `/app/data` or bake the index into the image.
- **AWS**: run the same Docker images on ECS/Fargate.

For a portfolio demo, Railway is usually simplest.
